# PathMNIST — ML Pipeline (Summative)

**Course:** ALU BSE — Machine Learning Pipeline

This notebook supports the rubric: **data acquisition**, **preprocessing**, **pre-trained MobileNetV2**, **optimization** (early stopping, Adam with low LR), and **≥4 metrics** (accuracy, loss, precision, recall, F1).

## 1. Data acquisition (non-tabular images)
We use **PathMNIST** from MedMNIST: 28×28 RGB histology patches, 9 classes.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, precision_recall_fscore_support
from medmnist import PathMNIST as MedPathMNIST

IMG_SIZE = 128
NUM_CLASSES = 9
SEED = 42
tf.keras.utils.set_random_seed(SEED)

train_raw = MedPathMNIST(split="train", download=True)
test_raw = MedPathMNIST(split="test", download=True)
print("Train samples:", len(train_raw), "Test samples:", len(test_raw))

## 2. EDA — three interpretable features
- **Class balance** — distribution of labels.
- **Mean channel intensity** — proxy for H&E staining / tissue type.
- **Gradient magnitude (edge energy)** — texture / boundary complexity.

In [ ]:
import matplotlib.pyplot as plt

labels = []
for i in range(len(train_raw)):
    _, y = train_raw[i]
    labels.append(int(np.asarray(y).reshape(-1)[0]))
labels = np.array(labels)
counts = np.bincount(labels, minlength=NUM_CLASSES)
plt.figure(figsize=(8,5))
plt.bar(range(NUM_CLASSES), counts)
plt.xlabel("Class id")
plt.ylabel("Count")
plt.title("Class distribution (train)")
plt.show()

sample_imgs = []
for c in range(NUM_CLASSES):
    idx = np.where(labels == c)[0][0]
    img, _ = train_raw[idx]
    sample_imgs.append(np.asarray(img))
stack = np.stack(sample_imgs, axis=0).astype(np.float32) / 255.0
mean_rgb = stack.reshape(-1, 3).mean(axis=0)
print("Global mean RGB (on one-per-class sample):", mean_rgb)

gx = np.abs(np.diff(stack, axis=2)).mean()
print("Mean gradient magnitude (proxy edges):", float(gx))

## 3. Preprocessing
Resize to 128×128, apply **MobileNetV2 `preprocess_input`**, train/val split.

In [ ]:
def to_batch(arr_x, arr_y, batch_size=64, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((arr_x, arr_y))
    if shuffle:
        ds = ds.shuffle(2048, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def load_arrays(dataset, limit=None):
    xs, ys = [], []
    n = len(dataset) if limit is None else min(limit, len(dataset))
    for i in range(n):
        img, lbl = dataset[i]
        x = np.asarray(img, dtype=np.uint8)
        if x.ndim == 2:
            x = np.stack([x, x, x], axis=-1)
        elif x.shape[-1] == 1:
            x = np.repeat(x, 3, axis=-1)
        y = int(np.asarray(lbl).reshape(-1)[0])
        xs.append(x)
        ys.append(y)
    return np.stack(xs), np.array(ys, dtype=np.int64)

N_TRAIN = 8000
N_TEST = 2000
x_train, y_train = load_arrays(train_raw, N_TRAIN)
x_test, y_test = load_arrays(test_raw, N_TEST)

def prep_tf(x, y):
    x = tf.cast(x, tf.float32)
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = preprocess_input(x)
    return x, y

val_split = 0.2
n_val = int(len(x_train) * val_split)
x_val, y_val = x_train[:n_val], y_train[:n_val]
x_train, y_train = x_train[n_val:], y_train[n_val:]

train_ds = to_batch(x_train, y_train).map(prep_tf, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = to_batch(x_val, y_val, shuffle=False).map(prep_tf, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = to_batch(x_test, y_test, shuffle=False).map(prep_tf, num_parallel_calls=tf.data.AUTOTUNE)
print(x_train.shape, y_train.shape)

## 4. Model — pre-trained MobileNetV2 + head
Optimization: **Adam**, **early stopping** on `val_loss`, small learning rate for fine-tuning.

In [ ]:
base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=[es])
model.summary()

## 5. Evaluation — ≥4 metrics
Accuracy, loss (from training), plus **precision, recall, F1** per class (macro averages).

In [ ]:
import pandas as pd

probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(probs, axis=1)
print(classification_report(y_test, y_pred, digits=3))
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, labels=list(range(NUM_CLASSES)))
pd.DataFrame({"precision": p, "recall": r, "f1": f1}).round(3)

## 6. Export model artifact
Save as **`.h5`** for the FastAPI service (`models/mobilenet_pathmnist.h5`).

In [ ]:
import os
os.makedirs("../models", exist_ok=True)
out = "../models/mobilenet_pathmnist.h5"
model.save(out)
print("Saved:", os.path.abspath(out))